# IBM Before the Breach — standalone Granite reranker + CVE investigator

Run all cells in order. This version loads `data/` directly and caches Granite vectors in `outputs/`. Current NIST descriptions may include post-snapshot updates, so Granite results remain exploratory.


In [1]:
# Standalone setup: load historical CVEs and create/cache Granite embeddings.
from pathlib import Path
import numpy as np
import pandas as pd
import re

DATA = Path("data")
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
FLAGS = ["exploitdb", "metasploit", "nuclei", "poc_github"]
HORIZON_DAYS = 90
assert DATA.exists(), "Expected a data/ folder next to this notebook."

vuln = pd.read_csv(DATA / "vulnerabilities.csv",
                   usecols=["cve_id", "published", "description"],
                   dtype={"cve_id": str, "description": str})
vuln["published"] = pd.to_datetime(vuln["published"], errors="coerce")
vuln = vuln.dropna(subset=["published"]).drop_duplicates("cve_id")

kev = pd.read_csv(DATA / "known_exploited.csv",
                  usecols=["cve_id", "date_added"])
kev["date_added"] = pd.to_datetime(kev["date_added"], errors="coerce")
kev = kev.dropna(subset=["date_added"]).groupby("cve_id", as_index=False)["date_added"].min()

signals = pd.concat([
    pd.read_csv(DATA / f"exploit_signals_{year}.csv",
                usecols=["month", "cve_id", "epss", *FLAGS])
    for year in (2024, 2025)
], ignore_index=True).drop_duplicates(["month", "cve_id"], keep="last")

hist = vuln[vuln["published"].dt.year.isin([2024, 2025])].copy()
hist["snapshot_month"] = hist["published"].dt.strftime("%Y-%m")
hist = hist.merge(signals, left_on=["cve_id", "snapshot_month"],
                  right_on=["cve_id", "month"], how="left", validate="one_to_one")
hist = hist.drop(columns="month")
hist = hist.merge(kev, on="cve_id", how="left", validate="one_to_one")
hist["snapshot_date"] = pd.to_datetime(hist["snapshot_month"] + "-01") + pd.offsets.MonthEnd(0)
hist = hist[hist["date_added"].isna() | (hist["date_added"] > hist["snapshot_date"])].copy()
hist = hist[hist["snapshot_date"] + pd.Timedelta(days=HORIZON_DAYS)
            <= pd.Timestamp("2025-12-31")].copy()
hist["label"] = (
    hist["date_added"].notna()
    & (hist["date_added"] > hist["snapshot_date"])
    & (hist["date_added"] <= hist["snapshot_date"] + pd.Timedelta(days=HORIZON_DAYS))
).astype("int8")

scoreboard_ids = pd.read_csv(DATA / "scoreboard_2026.csv", usecols=["cve_id"])
all_ids = pd.concat([hist[["cve_id", "description"]],
                     scoreboard_ids.merge(vuln[["cve_id", "description"]],
                                          on="cve_id", how="left", validate="one_to_one")],
                    ignore_index=True).drop_duplicates("cve_id").reset_index(drop=True)
id_to_ix = pd.Series(np.arange(len(all_ids)), index=all_ids["cve_id"])

HINDSIGHT = re.compile(
    r"\b(?:exploited in the wild|actively exploited|known exploited|added to (?:the )?kev|cisa kev|observed in attacks)\b",
    re.IGNORECASE,
)
def clean_text(s):
    return HINDSIGHT.sub(" ", str(s) if pd.notna(s) else "")[:1800]

cache_vectors = OUT / "granite_vectors.npy"
cache_ids = OUT / "granite_vector_ids.csv"
ids_now = all_ids["cve_id"].astype(str).reset_index(drop=True)
cache_ok = False
if cache_vectors.exists() and cache_ids.exists():
    old_ids = pd.read_csv(cache_ids, dtype=str)["cve_id"]
    cache_ok = old_ids.equals(ids_now)

if cache_ok:
    vec = np.load(cache_vectors, mmap_mode="r")
    print("Reusing cached Granite embeddings")
else:
    from sentence_transformers import SentenceTransformer
    encoder = SentenceTransformer("ibm-granite/granite-embedding-30m-english")
    vec = encoder.encode(all_ids["description"].map(clean_text).tolist(),
                         batch_size=64, show_progress_bar=True,
                         normalize_embeddings=True,
                         convert_to_numpy=True).astype("float32")
    np.save(cache_vectors, vec)
    ids_now.to_frame("cve_id").to_csv(cache_ids, index=False)
    print("Saved embeddings for future reruns")

assert vec.shape == (len(all_ids), 384), f"Unexpected embedding shape: {vec.shape}"
print(f"Ready: {len(hist):,} historical CVEs, {hist.label.sum()} positives, "
      f"{len(scoreboard_ids):,} scoreboard CVEs, vectors {vec.shape}")


c:\Users\shany\Documents\PatchRader\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10300.50it/s]
This model was loaded in torch.bfloat16 on CPU. Depending on your CPU, inference may be significantly slower than with torch.float32. Consider comparing inference speeds by loading with `model_kwargs={"dtype": torch.float32}`.
Batches: 100%|██████████| 2310/2310 [14:49<00:00,  2.60it/s]


Saved embeddings for future reruns
Ready: 75,149 historical CVEs, 73 positives, 72,670 scoreboard CVEs, vectors (147819, 384)


In [7]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

DATA = Path("data")
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
SEED = 42
WEIGHT_GRANITE = 0.10  # fixed before evaluating these folds
HORIZON_DAYS = 90
BUDGET_FRACTION = 500 / 72670
EPOCHS = 10  # identical to prior rolling-head configuration
FLAGS = ["exploitdb", "metasploit", "nuclei", "poc_github"]
FOLDS = [
    ("2025-Q1", "2025-01-01", "2025-04-01"),
    ("2025-Q2", "2025-04-01", "2025-07-01"),
    ("2025-Q3 (previously inspected)", "2025-07-01", "2025-10-01"),
]

assert all(x in globals() for x in ["hist", "vec", "id_to_ix"]), "Run the standalone setup cell above first."
assert set(["cve_id", "snapshot_month", "label", "epss", *FLAGS]).issubset(hist.columns)
assert not hist.cve_id.duplicated().any(), "Expected one publish-month record per CVE; don't use repeated monthly snapshots here."
assert all(hist.cve_id.isin(id_to_ix.index)), "Missing Granite embedding IDs for historical observations."
print("Historic:", hist.shape, "Granite vectors:", vec.shape)


Historic: (75149, 12) Granite vectors: (147819, 384)


In [3]:
# Predictors and deterministic, stable order

def epss_array(frame):
    return pd.to_numeric(frame["epss"], errors="coerce").fillna(0).to_numpy(float)

def exploit_array(frame):
    return frame[FLAGS].fillna(0).gt(0).any(axis=1).to_numpy(dtype=int)

def group_percentiles(values, groups):
    """Ranks score WITHIN exploit flag strata; stronger values -> larger percentile."""
    values = np.asarray(values, dtype=float)
    groups = np.asarray(groups)
    out = np.zeros(len(values), dtype=float)
    for g in np.unique(groups):
        ix = np.flatnonzero(groups == g)
        ranks = pd.Series(values[ix]).rank(method="average").to_numpy()
        out[ix] = (ranks - 1) / max(len(ix) - 1, 1)
    return out

def final_order(frame, text_scores=None, weight=0.0):
    epss = epss_array(frame)
    flags = exploit_array(frame)
    ep = group_percentiles(epss, flags)
    if text_scores is None or weight == 0:
        blended = ep
    else:
        hp = group_percentiles(text_scores, flags)
        blended = (1-weight)*ep + weight*hp
    # Stable tie-breaking: stronger exploit indicator, then blend, then EPSS, then CVE ID.
    order = np.lexsort((frame["cve_id"].astype(str).to_numpy(), -epss, -blended, -flags))
    return order, blended

def ranking_scores(order):
    scores = np.empty(len(order), dtype=float)
    scores[order] = np.arange(len(order), 0, -1)
    return scores

def report(frame, order, model, fold):
    y = frame["label"].to_numpy(dtype=int)
    k = max(1, round(len(frame) * BUDGET_FRACTION))
    hits = int(y[order[:k]].sum())
    sc = ranking_scores(order)
    return dict(fold=fold, model=model, n=len(frame), positives=int(y.sum()), k=k,
                hits_at_k=hits, precision_at_k=hits/k,
                capture_rate_at_k=hits/max(int(y.sum()),1),
                average_precision=average_precision_score(y,sc) if y.sum() else float('nan'),
                roc_auc=roc_auc_score(y,sc) if y.min()!=y.max() else float('nan'))

class GraniteRiskHead(nn.Module):
    def __init__(self, dim=384):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim,32), nn.ReLU(), nn.Dropout(.3), nn.Linear(32,1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

def fit_head(train_embeddings, y, seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = GraniteRiskHead(train_embeddings.shape[1])
    n_pos = int(np.sum(y))
    if n_pos < 1: raise ValueError("No train positives")
    weight = min((len(y)-n_pos)/n_pos,100.0)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(weight)))
    opt = torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-2)
    loader = DataLoader(TensorDataset(torch.from_numpy(train_embeddings.astype('float32')),
                  torch.tensor(y,dtype=torch.float32)),batch_size=512,shuffle=True)
    for _ in range(EPOCHS):
        model.train()
        for xb,yb in loader:
            opt.zero_grad(); loss=loss_fn(model(xb),yb); loss.backward(); opt.step()
    model.eval()
    return model

def predict_head(model, arr, batch_size=4096):
    predictions=[]
    with torch.no_grad():
        for start in range(0,len(arr),batch_size):
            xx=torch.from_numpy(np.asarray(arr[start:start+batch_size],dtype='float32'))
            predictions.append(torch.sigmoid(model(xx)).cpu().numpy())
    return np.concatenate(predictions) if predictions else np.array([])


In [4]:
# Rolling-origin comparison. Historical 90-day labels must mature before fold start.
frame=hist.copy()
frame['snapshot_date']=pd.to_datetime(frame['snapshot_month'].astype(str)+'-01', errors='coerce') + pd.offsets.MonthEnd(0)
frame=frame.dropna(subset=['snapshot_date','cve_id','label']).copy()
records=[]
head_scores_by_fold={}
for fold,start_str,end_str in FOLDS:
    start,end=pd.Timestamp(start_str),pd.Timestamp(end_str)
    tr=frame.loc[(frame.snapshot_date + pd.Timedelta(days=HORIZON_DAYS)) < start].reset_index(drop=True)
    va=frame.loc[(frame.snapshot_date>=start)&(frame.snapshot_date<end)].reset_index(drop=True)
    if tr.label.sum()<5 or va.label.sum()<1:
        print('Skipping',fold,'insufficient positives'); continue
    tr_idx=id_to_ix.loc[tr.cve_id].to_numpy()
    va_idx=id_to_ix.loc[va.cve_id].to_numpy()
    scaler=StandardScaler()
    xtr=scaler.fit_transform(vec[tr_idx].astype('float32')).astype('float32')
    xva=scaler.transform(vec[va_idx].astype('float32')).astype('float32')
    model=fit_head(xtr,tr.label.to_numpy(dtype=int))
    head=predict_head(model,xva)
    baseline,_=final_order(va)
    reranker,_=final_order(va,head,weight=WEIGHT_GRANITE)
    records += [report(va,baseline,'Exploit-first EPSS',fold),
                report(va,reranker,'Exploit-first + 10% Granite',fold)]
    head_scores_by_fold[fold]=head
    print(f'{fold}: training={len(tr):,} ({int(tr.label.sum())} positives), '
          f'validation={len(va):,} ({int(va.label.sum())} positives), '
          f'baseline hits={records[-2]["hits_at_k"]}, Granite hits={records[-1]["hits_at_k"]}')

fold_results=pd.DataFrame(records)
if fold_results.empty: raise RuntimeError('No eligible folds')
from IPython.display import display
display(fold_results)
summary=fold_results.groupby('model').agg(total_k=('k','sum'),total_hits=('hits_at_k','sum'),
               mean_ap=('average_precision','mean'),mean_auc=('roc_auc','mean'))
summary['pooled_precision_at_k']=summary.total_hits/summary.total_k
pivot=fold_results.pivot(index='fold',columns='model',values='hits_at_k')
display(summary); display(pivot)
summary.to_csv(OUT/'granite_rerank_summary.csv')
fold_results.to_csv(OUT/'granite_rerank_folds.csv',index=False)

# Predeclared conservative rule: every fold must avoid losing hits AND total must improve.
use_granite=(summary.loc['Exploit-first + 10% Granite','total_hits'] >
             summary.loc['Exploit-first EPSS','total_hits'] and
             (pivot['Exploit-first + 10% Granite'] >= pivot['Exploit-first EPSS']).all())
print('SELECTED:', 'Granite 10% reranker (EXPLORATORY, description leakage warning)' if use_granite
      else 'Exploit-first EPSS baseline')


2025-Q1: training=28,839 (20 positives), validation=11,913 (14 positives), baseline hits=3, Granite hits=3
2025-Q2: training=39,868 (32 positives), validation=11,666 (14 positives), baseline hits=6, Granite hits=6
2025-Q3 (previously inspected): training=51,781 (46 positives), validation=11,702 (13 positives), baseline hits=2, Granite hits=2


,fold,model,n,positives,k,hits_at_k,precision_at_k,capture_rate_at_k,average_precision,roc_auc
0,2025-Q1,Exploit-first EPSS,11913,14,82,3,0.036585,0.214286,0.087504,0.697117
1,2025-Q1,Exploit-first + 10% Granite,11913,14,82,3,0.036585,0.214286,0.088255,0.736016
2,2025-Q2,Exploit-first EPSS,11666,14,80,6,0.075000,0.428571,0.134961,0.785598
3,2025-Q2,Exploit-first + 10% Granite,11666,14,80,6,0.075000,0.428571,0.090720,0.780344
4,2025-Q3 (previously inspected),Exploit-first EPSS,11702,13,81,2,0.024691,0.153846,0.011312,0.835388
5,2025-Q3 (previously inspected),Exploit-first + 10% Granite,11702,13,81,2,0.024691,0.153846,0.012443,0.845766


,total_k,total_hits,mean_ap,mean_auc,pooled_precision_at_k
model,,,,,
Exploit-first + 10% Granite,243,11,0.063806,0.787375,0.045267
Exploit-first EPSS,243,11,0.077926,0.772701,0.045267


model,Exploit-first + 10% Granite,Exploit-first EPSS
fold,,
2025-Q1,3,3
2025-Q2,6,6
2025-Q3 (previously inspected),2,2


SELECTED: Exploit-first EPSS baseline


In [5]:
# Fit final Granite head only on matured historical 90-day labels and score scoreboard.
# The Granite model is only used for experimental ranking. Baseline will still export regardless.
scoreboard=pd.read_csv(DATA/'scoreboard_2026.csv')
assert len(scoreboard)==72670 and scoreboard.cve_id.is_unique
assert scoreboard.cve_id.isin(id_to_ix.index).all(), 'Run previous Granite encoding step including 2026 scoreboard CVEs.'
for col in FLAGS: scoreboard[col]=pd.to_numeric(scoreboard[col],errors='coerce').fillna(0)
scoreboard['epss']=pd.to_numeric(scoreboard['epss'],errors='coerce').fillna(0)

matured=frame.loc[(frame.snapshot_date+pd.Timedelta(days=HORIZON_DAYS))<=pd.Timestamp('2025-12-31')].copy()
train_ix=id_to_ix.loc[matured.cve_id].to_numpy()
score_ix=id_to_ix.loc[scoreboard.cve_id].to_numpy()
scaler=StandardScaler()
X_final=scaler.fit_transform(vec[train_ix].astype('float32')).astype('float32')
X_score=scaler.transform(vec[score_ix].astype('float32')).astype('float32')
final_head=fit_head(X_final,matured.label.to_numpy(dtype=int))
scoreboard['granite_head_score']=predict_head(final_head,X_score)
base_order,base_blend=final_order(scoreboard)
ml_order,ml_blend=final_order(scoreboard,scoreboard['granite_head_score'],WEIGHT_GRANITE)
scoreboard['baseline_rank']=pd.Series(np.arange(len(scoreboard)),index=base_order).sort_index().to_numpy()+1
scoreboard['granite_candidate_rank']=pd.Series(np.arange(len(scoreboard)),index=ml_order).sort_index().to_numpy()+1
scoreboard['any_exploit']=exploit_array(scoreboard)
scoreboard['exploit_source_count']=scoreboard[FLAGS].gt(0).sum(axis=1)
scoreboard['selected_rank']=scoreboard['granite_candidate_rank'] if use_granite else scoreboard['baseline_rank']
scoreboard['selection_method']='Granite-head reranker (exploratory)' if use_granite else 'Exploit-first EPSS'
scoreboard.sort_values('baseline_rank')[['cve_id']].head(500).to_csv(OUT/'top500_baseline.csv',index=False)
scoreboard.sort_values('granite_candidate_rank')[['cve_id']].head(500).to_csv(OUT/'top500_granite_candidate.csv',index=False)
scoreboard.sort_values('selected_rank')[['cve_id']].head(500).to_csv(OUT/'top500_selected.csv',index=False)
scoreboard.sort_values('selected_rank').to_csv(OUT/'ranked_2026.csv',index=False)
print('Selection:',scoreboard.selection_method.iloc[0])
print('Saved outputs/top500_baseline.csv, top500_granite_candidate.csv, top500_selected.csv, ranked_2026.csv')
print('Difference in candidate top-500 sets:',len(set(scoreboard.nsmallest(500,'baseline_rank').cve_id) ^ set(scoreboard.nsmallest(500,'granite_candidate_rank').cve_id)))
display(scoreboard.nsmallest(20,'selected_rank')[['cve_id','epss','any_exploit','granite_head_score','baseline_rank','granite_candidate_rank','selected_rank']])


Selection: Exploit-first EPSS
Saved outputs/top500_baseline.csv, top500_granite_candidate.csv, top500_selected.csv, ranked_2026.csv
Difference in candidate top-500 sets: 40


,cve_id,epss,any_exploit,granite_head_score,baseline_rank,granite_candidate_rank,selected_rank
62040,CVE-2026-8037,0.84793,1,0.010593,1,10,1
1596,CVE-2022-25369,0.63500,1,0.001603,2,29,2
2072,CVE-2024-45257,0.59548,1,0.004553,3,21,3
40258,CVE-2026-49160,0.53829,1,0.037284,4,5,4
2907,CVE-2025-14558,0.53599,1,0.001861,5,30,5
9550,CVE-2026-10523,0.51870,1,0.207809,6,1,6
2104,CVE-2024-51092,0.44112,1,0.025938,7,8,7
24158,CVE-2026-27771,0.43068,1,0.000093,8,72,8
25345,CVE-2026-29058,0.42990,1,0.254299,9,2,9
20349,CVE-2026-2329,0.41137,1,0.048263,10,7,10


## Semantic retrieval dataset for the production investigation app

Generate a **small local index** for the top-500 plus historical confirmed KEV CVEs; this is much faster than embedding the entire 401k-row NIST file again. Use exactly the same Granite vectors already computed. Similarity is descriptive, **not** evidence of an active attack or a historical prediction feature. KEV descriptions and today's NIST descriptions are retrospective and should be labeled as such in the app.

In [6]:
vuln_text=pd.read_csv(DATA/'vulnerabilities.csv',usecols=['cve_id','description','published'],dtype={'cve_id':str,'description':str})
kev=pd.read_csv(DATA/'known_exploited.csv',usecols=['cve_id','date_added'],dtype={'cve_id':str})
kev['date_added']=pd.to_datetime(kev['date_added'],errors='coerce')
# Only actual historical cases included in the KEV data (through 2025-12-31)
kev=kev.drop_duplicates('cve_id')
priority_ids=scoreboard.nsmallest(500,'selected_rank').cve_id.tolist()
index_ids=list(dict.fromkeys(priority_ids + kev.cve_id.tolist()))
index_ids=[c for c in index_ids if c in set(vuln_text.cve_id)]
existing=[c for c in index_ids if c in id_to_ix.index]
missing=[c for c in index_ids if c not in id_to_ix.index]
emb_map={c: np.asarray(vec[id_to_ix.loc[c]],dtype='float32') for c in existing}
if missing:
    from sentence_transformers import SentenceTransformer
    import re
    encoder_index=SentenceTransformer('ibm-granite/granite-embedding-30m-english')
    hindsight_regex=re.compile(r'(?:exploited in the wild|actively exploited|known exploited|added to (?:the )?kev|cisa kev|observed in attacks)',re.I)
    descriptions=vuln_text.drop_duplicates('cve_id').set_index('cve_id').loc[missing,'description']
    clean=[hindsight_regex.sub(' ',str(s) if pd.notna(s) else '')[:1800] for s in descriptions]
    vectors_new=encoder_index.encode(clean,batch_size=64,normalize_embeddings=True,
                                   show_progress_bar=True,convert_to_numpy=True)
    emb_map.update({c:v for c,v in zip(missing,vectors_new)})
index_vectors=np.stack([emb_map[c] for c in index_ids]).astype('float32')
index_vectors/=np.maximum(np.linalg.norm(index_vectors,axis=1,keepdims=True),1e-12)
np.savez_compressed(OUT/'semantic_index.npz',cve_id=np.asarray(index_ids),vectors=index_vectors)
meta=pd.DataFrame({'cve_id':index_ids}).merge(vuln_text.drop_duplicates('cve_id'),how='left',on='cve_id')
meta=meta.merge(kev,how='left',on='cve_id')
meta['is_historical_kev']=meta.date_added.notna()
meta.to_csv(OUT/'semantic_metadata.csv',index=False)
print('Semantic index:',len(index_ids),'CVEs, embeddings:',index_vectors.shape)
print('App inputs ready in:',OUT.resolve())


Batches: 100%|██████████| 22/22 [00:10<00:00,  2.15it/s]


Semantic index: 1984 CVEs, embeddings: (1984, 384)
App inputs ready in: C:\Users\shany\Documents\PatchRader\outputs


## Run the app

Save `cve_investigator.py` in your project root (provided separately) and then:

```bash
pip install -r requirements_investigator.txt
streamlit run cve_investigator.py
```

The app uses the embedding vectors generated above for nearest-neighbor search. It shows **rule-based, traceable explanations** of why each CVE was ranked and retrieves semantically similar historical CVEs. It does **not** generate unsupported exploit claims, automate patch deployment, or assume KEV is a complete list of exploits.

**Submission note:** `outputs/top500_selected.csv` is the chosen artifact and has exactly one `cve_id` column. Copy it to `top500.csv` for Devpost if desired. Inspect `granite_rerank_folds.csv` first.